# GoldenCheetah 데이터 탐색

## 분석 목적

GoldenCheetah 공개 데이터가 훈련 분석과 오늘의 훈련 추천 프로젝트에
사용할 수 있는지 확인한다.

## 현재까지 확인한 내용

- 전체 운동 기록: 731개
- 자전거 운동: 592개
- 파워 센서 포함: 469개
- 심박 센서 포함: 465개
- 파워와 심박 모두 포함: 373개
- 라이딩마다 센서와 요약 지표 구성이 다름

In [2]:
import sys
from pathlib import Path

import json

import pandas as pd

project_root = Path("..").resolve()

print("Python 경로:", sys.executable)
print("프로젝트 경로:", project_root)
print("pandas 버전:", pd.__version__)

json_path = (
    project_root
    / "data"
    / "raw"
    / "033874ce-e20d-44ba-9cc9-125030b6662f"
    / "{033874ce-e20d-44ba-9cc9-125030b6662f}.json"
)

Python 경로: /Users/hooni/Documents/ChatGPT/Cycling App/.venv/bin/python
프로젝트 경로: /Users/hooni/Documents/ChatGPT/Cycling App
pandas 버전: 3.0.5


## 1. JSON 데이터 불러오기

JSON 파일을 Python 자료형으로 불러온다.
전체 운동 중 `Bike` 기록만 선택하여 pandas 분석에 사용한다.

In [3]:
with json_path.open("r", encoding="utf-8") as file:
    cycling_data = json.load(file)

rides = cycling_data["RIDES"]

bike_rides = [ride for ride in rides if ride["sport"] == "Bike"]

print("전체 운동 수:", len(rides))
print("자전거 운동 수:", len(bike_rides))

전체 운동 수: 731
자전거 운동 수: 592


라이드 기록의 `data`는 15자리의 대문자 알파벳 문자열을 값으로 가지는데, 해당 운동에 어떤 센서 데이터가 포함되어 있는지 나타낸다.

| 문자 | 데이터 |
|---|---|
| `T` | 시간 |
| `D` | 거리 |
| `S` | 속도 |
| `P` | 파워 |
| `H` | 심박수 |
| `C` | 케이던스 |
| `N` | 토크 |
| `A` | 고도 |
| `G` | GPS |
| `L` | 경사도 |
| `W` | 풍속 |
| `E` | 온도 |
| `V` | 좌우 페달 데이터 |
| `O` | 근육 산소 관련 데이터 |
| `R` | Garmin 러닝 다이내믹스 |

In [4]:
first_ride = bike_rides[0]

print(first_ride.keys())
print("첫 라이드의 날짜", first_ride["date"])
print("첫 라이드 기록의 센서 목록:", first_ride["data"])

dict_keys(['date', 'data', 'sport', 'METRICS'])
첫 라이드의 날짜 2005/06/25 14:26:00 UTC
첫 라이드 기록의 센서 목록: TDS-H--A-L-----


첫 라이드의 경우 `TDS-H--A-L-----`로, 시간, 거리, 속도, 심박수, 고도, 경사도 데이터를 포함하고 있다.

## 2. 자전거 기록을 표로 변환

592개의 자전거 기록은 딕셔너리로 구성되어 있다.
pandas의 `DataFrame`을 이용하여 행과 열로 구성된 표로 변환한다.

In [5]:
bike_raw_df = pd.DataFrame(bike_rides)

print("자료형", type(bike_raw_df))
print("표 크기:", bike_raw_df.shape)
print("열 이름", bike_raw_df.columns)

bike_raw_df.head()

자료형 <class 'pandas.DataFrame'>
표 크기: (592, 5)
열 이름 Index(['date', 'data', 'sport', 'METRICS', 'XDATA'], dtype='str')


,date,data,sport,METRICS,XDATA
0,2005/06/25 14:26:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
1,2005/06/27 08:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
2,2005/06/28 17:44:00 UTC,TDS-HC-A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
3,2005/07/06 18:02:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
4,2005/07/10 09:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN


- 표는 592개의 행과 5개의 열로 구성되어 있다.
- 기본 열은 `date`, `data`, `sport`, `METRICS`, `XDATA`이다.
- `METRICS`는 아직 하나의 딕셔너리로 저장되어 있다.
- `XDATA`는 일부 라이딩에만 존재하므로 대부분 결측값으로 표시된다.

## 3. METRICS 펼치기

각 라이딩의 `METRICS`에는 운동 시간, 거리, 파워, 심박수 등
여러 요약 지표가 딕셔너리 형태로 저장되어 있다.

`METRICS`의 각 키를 DataFrame의 개별 열로 변환한다.

In [6]:
metrics_df = pd.json_normalize(bike_raw_df["METRICS"])

print("METRICS 표 크기", metrics_df.shape)
print("앞쪽 열 10개:")
print(metrics_df.columns[:10])

METRICS 표 크기 (592, 228)
앞쪽 열 10개:
Index(['a_skiba_variability_index', 'a_coggam_variability_index', 'ride_count',
       'workout_time', 'time_riding', 'total_distance', 'climb_rating',
       'athlete_weight', 'elevation_gain', 'elevation_loss'],
      dtype='str')


In [7]:
sample_columns = [
    "workout_time",
    'time_riding',
    "total_distance",
    'elevation_gain',
    "average_power",
    "average_hr",
    "coggan_tss",
    "coggan_if",
]

metrics_df[sample_columns].head()

,workout_time,time_riding,total_distance,elevation_gain,average_power,average_hr,coggan_tss,coggan_if
0,4800.00000,4780.00000,35.32750,367.00000,NaN,"[146.91667, 960.00000]",NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,NaN,"[124.97267, 6476.00000]",NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,NaN,"[157.04592, 2156.00000]",NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,NaN,"[120.05660, 1272.00000]",NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,NaN,"[146.42735, 936.00000]",NaN,NaN


- 592개 자전거 기록의 `METRICS`를 펼치자 228개의 지표가 나타났다.
- 라이딩마다 포함된 지표가 달라 전체 지표 수가 많아졌다.
- 이전 .py 파일에서 보았듯이, 평균 파워와 평균 심박은 일부 행에서 리스트 형태로 저장되어 있다.
- 파워 센서가 없는 초기 라이딩에서는 파워, TSS, IF가 결측값으로 표시된다.
- 228개 지표를 모두 사용하지 않고 분석 목적에 필요한 지표만 선택해야 한다.

## 4. 분석에 사용할 지표 선택

228개의 `METRICS` 지표들은 모두 서로 다른 센서 데이터가 아니라, 기본 센서값을
여러 계산 방식으로 요약한 지표도 포함하고 있다.

예를 들어 파워 관련 지표에는 다음과 같은 계산 계열이 존재한다.

- `coggan_`: Coggan 방식의 파워 강도와 훈련 부하 지표
- `skiba_`: Skiba 방식의 파워 강도와 훈련 부하 지표
- `a_`: 고도의 영향을 반영한 보정 지표

비슷한 의미의 지표를 모두 사용하면 정보가 중복될 수 있으므로,
첫 번째 분석에서는 해석하기 쉬운 기본 지표와 Coggan 계열을 우선 사용한다.

### 기본 정보

- `date`: 운동 날짜
- `data`: 기록된 센서 종류
- `sport`: 운동 종류

### 운동량

- `workout_time`: 전체 운동 시간
- `time_riding`: 실제 이동 시간
- `total_distance`: 총거리
- `elevation_gain`: 누적 상승고도
- `average_speed`: 평균 속도

### 파워

- `average_power`: 평균 파워
- `coggan_np`: 변동성을 고려한 대표 파워
- `max_power`: 최대 파워
- `cp_setting`: 해당 시점의 기준 파워

### 심박과 케이던스

- `average_hr`: 평균 심박수
- `max_heartrate`: 최대 심박수
- `average_cad`: 평균 케이던스
- `max_cadence`: 최대 케이던스

### 훈련 강도와 부하

- `coggan_if`: 기준 파워 대비 운동 강도
- `coggan_tss`: 운동 시간과 강도를 반영한 훈련 부하

`skiba_` 계열과 `a_` 계열은 원본에 보존하고, 이후 계산 방식과
고도 영향을 비교할 필요가 생기면 별도로 분석한다.

In [8]:
selected_metric_columns = [
    "workout_time",     # 전체 운동 시간
    "time_riding",      # 실제 이동 시간
    "total_distance",   # 총거리
    "elevation_gain",   # 획득 고도
    "average_speed",    # 평균 속도
    "average_power",    # 평균 파워
    "max_power",        # 최대 파워
    "coggan_np",        # NP
    "cp_setting",       # 해당 시점의 기준 파워
    "average_hr",       # 평균 심박수
    "max_heartrate",    # 최대 심박수
    "average_cad",      # 평균 케이던스
    "max_cadence",      # 최대 케이던스
    "coggan_if",        # IF
    "coggan_tss",       # TSS
]

selected_metrics_df = metrics_df[selected_metric_columns].copy()

print(selected_metrics_df.shape)
selected_metrics_df.head()

(592, 15)


,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


In [9]:
metadata_df = bike_raw_df[
    ["data", "date", "sport"]
].copy()

analysis_df = pd.concat(
    [metadata_df, selected_metrics_df],
    axis=1
)

print(analysis_df.shape)
analysis_df.head()

(592, 18)


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


In [10]:
print(analysis_df.dtypes)

data                 str
date                 str
sport                str
workout_time         str
time_riding          str
total_distance       str
elevation_gain       str
average_speed        str
average_power     object
max_power            str
coggan_np         object
cp_setting           str
average_hr        object
max_heartrate        str
average_cad       object
max_cadence          str
coggan_if         object
coggan_tss           str
dtype: object


In [11]:
for column in selected_metric_columns:
    values = analysis_df[column].dropna()
    first_value = values.iloc[0]

    print(
        column,
        "| 값:", first_value,
        "| 자료형:", type(first_value).__name__
    )

workout_time | 값: 4800.00000 | 자료형: str
time_riding | 값: 4780.00000 | 자료형: str
total_distance | 값: 35.32750 | 자료형: str
elevation_gain | 값: 367.00000 | 자료형: str
average_speed | 값: 26.60649 | 자료형: str
average_power | 값: ['188.85205', '12808.00000'] | 자료형: list
max_power | 값: 611.00000 | 자료형: str
coggan_np | 값: ['237.99715', '16138.08000'] | 자료형: list
cp_setting | 값: 200.00000 | 자료형: str
average_hr | 값: ['146.91667', '960.00000'] | 자료형: list
max_heartrate | 값: 184.00000 | 자료형: str
average_cad | 값: ['88.72929', '1919.00000'] | 자료형: list
max_cadence | 값: 98.00000 | 자료형: str
coggan_if | 값: ['0.86544', '16138.08000'] | 자료형: list
coggan_tss | 값: 335.75887 | 자료형: str


In [12]:
for column in selected_metric_columns:
    type_counts = (
        analysis_df[column]
        .dropna()
        .map(type)
        .value_counts()
    )

    print(f"{type_counts}\n")

workout_time
<class 'str'>    591
Name: count, dtype: int64

time_riding
<class 'str'>    574
Name: count, dtype: int64

total_distance
<class 'str'>    563
Name: count, dtype: int64

elevation_gain
<class 'str'>    496
Name: count, dtype: int64

average_speed
<class 'str'>    563
Name: count, dtype: int64

average_power
<class 'list'>    469
Name: count, dtype: int64

max_power
<class 'str'>    469
Name: count, dtype: int64

coggan_np
<class 'list'>    469
Name: count, dtype: int64

cp_setting
<class 'str'>    592
Name: count, dtype: int64

average_hr
<class 'list'>    465
<class 'str'>       3
Name: count, dtype: int64

max_heartrate
<class 'str'>    465
Name: count, dtype: int64

average_cad
<class 'list'>    532
Name: count, dtype: int64

max_cadence
<class 'str'>    532
Name: count, dtype: int64

coggan_if
<class 'list'>    470
Name: count, dtype: int64

coggan_tss
<class 'str'>    490
Name: count, dtype: int64



In [13]:
average_hr_is_string = analysis_df["average_hr"].map(type) == str

analysis_df.loc[
    average_hr_is_string,
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
        "workout_time",
    ]
]

,date,data,average_hr,max_heartrate,workout_time
392,2012/01/04 15:35:51 UTC,---------------,148.00000,NaN,4200.00000
393,2012/01/05 12:32:51 UTC,---------------,145.00000,NaN,5820.00000
394,2012/01/06 13:22:53 UTC,---------------,145.00000,NaN,2700.00000


### 자료형 확인 결과

`average_hr`의 유효한 값 대부분은 리스트이지만, 3개는 문자열로
저장되어 있다.

해당 3개 라이드는 `data`에 센서 기록이 표시되어 있지 않고
최대 심박수도 비어 있다. 따라서 시계열 심박 데이터에서 계산된 값이
아니라, 수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

지금은 삭제하지 않고 이후 데이터 품질을 분류할 때 별도로 구분한다.

## 5. 지표의 자료형 확인과 변환

선택한 지표의 실제 값을 확인한 결과, 숫자가 다음 두 가지 형태로
저장되어 있었다.

- 숫자를 나타내는 문자열: `"4316.00000"`
- 지표값과 관측 정보를 담은 리스트: `["145.75371", "4316.00000"]`

리스트의 첫 번째 요소는 분석에 사용할 지표값이고, 두 번째 요소는
평균 계산에 사용된 관측값 수 또는 가중치 정보로 보인다.

`average_hr`에서는 대부분의 값이 리스트였지만 3개는 문자열이었다.
이 3개 라이드는 센서 정보를 나타내는 `data`가 비어 있고 최대 심박수도
기록되어 있지 않았다. 따라서 센서 시계열에서 계산된 값이 아니라
수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

원본 구조를 보존하기 위해 `analysis_df`는 변경하지 않는다.
대신 `cleaned_df`를 복사하여 다음과 같이 변환한다.

- 리스트는 첫 번째 요소를 사용한다.
- 숫자 형태의 문자열은 실제 숫자로 변환한다.
- 변환할 수 없는 값은 결측값으로 처리한다.
- 리스트의 두 번째 요소는 필요할 경우 원본 데이터에서 다시 확인한다.

In [14]:
def extract_metric_value(value):
    if isinstance(value, list):
        value = value[0]

    return pd.to_numeric(value, errors="coerce")

In [15]:
cleaned_df = analysis_df.copy()

for column in selected_metric_columns:
    cleaned_df[column] = cleaned_df[column].map(
        extract_metric_value
    )

print(cleaned_df.dtypes)
display(cleaned_df.head())

data                  str
date                  str
sport                 str
workout_time      float64
time_riding       float64
total_distance    float64
elevation_gain    float64
average_speed     float64
average_power     float64
max_power         float64
coggan_np         float64
cp_setting        float64
average_hr        float64
max_heartrate     float64
average_cad       float64
max_cadence       float64
coggan_if         float64
coggan_tss        float64
dtype: object


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.0,4780.0,35.3275,367.0,26.60649,NaN,NaN,NaN,200.0,146.91667,184.0,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.0,6325.0,35.0140,467.5,19.92892,NaN,NaN,NaN,200.0,124.97267,182.0,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.0,2156.0,17.6530,92.0,29.71052,NaN,NaN,NaN,200.0,157.04592,168.0,88.72929,98.0,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.0,6320.0,31.0495,512.0,17.68642,NaN,NaN,NaN,200.0,120.05660,182.0,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.0,4660.0,32.7700,471.0,25.31588,NaN,NaN,NaN,200.0,146.42735,177.0,NaN,NaN,NaN,NaN


## 6. 선택한 지표의 결측값 확인

숫자로 변환한 `cleaned_df`를 이용하여 각 후보 지표의 결측 개수와
결측 비율을 확인한다.

결측값의 분포를 통해 실제 분석에 사용할 수 있는 지표와
별도의 품질 기준이 필요한 지표를 판단한다.

In [18]:
missing_count = cleaned_df[selected_metric_columns].isna().sum()
missing_ratio = cleaned_df[selected_metric_columns].isna().mean() * 100

missing_summary_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio": missing_ratio,
})

missing_summary_df.round(2)

,missing_count,missing_ratio
workout_time,1,0.17
time_riding,18,3.04
total_distance,29,4.90
elevation_gain,96,16.22
average_speed,29,4.90
average_power,123,20.78
max_power,123,20.78
coggan_np,123,20.78
cp_setting,0,0.00
average_hr,124,20.95


### 결측값 확인 결과

파워와 심박 관련 지표는 전체 자전거 기록의 약 20%에서 결측값으로
나타났다. 모든 라이딩에 파워 미터와 심박 센서가 사용된 것은 아니기
때문으로 보인다.

IF도 파워 지표와 비슷한 결측 비율을 보였다. 반면 TSS는 파워보다
결측 비율이 낮았다. 일부 TSS는 파워 센서 데이터로부터 자동 계산된
값이 아니라 수동으로 입력했거나 별도의 방식으로 추정한 값일 가능성이
있다.

따라서 결측값이 있는 라이드를 모두 삭제하지 않고, 분석 목적과
센서 유무에 따라 사용할 라이드를 구분할 필요가 있다.